In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import re
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np

# Example: create dummy predictions
# Replace `test.csv` with your actual test file if needed

df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(df.shape)
print(test_df.shape)

(2000, 8)
(500, 7)


In [3]:
df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df["prompt"].nunique()

1758

In [6]:
df["answer"].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df["id"].duplicated().sum()

np.int64(0)

In [9]:
df_cols = df.select_dtypes(exclude="number").columns.tolist()

In [10]:
# The columns we want to analyze
text_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']

# List to store our computed statistics
stats = []

for col in text_cols:
    # Convert to string to handle any accidental NaN/null values gracefully
    col_data = df[col].astype(str)
    
    # Calculate lengths
    char_lengths = col_data.str.len()
    word_counts = col_data.str.split().str.len()
    
    # Append the metrics to our list
    stats.append({
        'Column': col,
        'Avg Char Length': round(char_lengths.mean(), 2),
        'Avg Word Count': round(word_counts.mean(), 2),
        'Min Char Length': char_lengths.min(),
        'Max Char Length': char_lengths.max(),
        'Min Word Count': word_counts.min(),
        'Max Word Count': word_counts.max()
    })

# Convert to a DataFrame for a clean, tabular display
eda_df = pd.DataFrame(stats)
print(eda_df.to_markdown(index=False))

| Column   |   Avg Char Length |   Avg Word Count |   Min Char Length |   Max Char Length |   Min Word Count |   Max Word Count |
|:---------|------------------:|-----------------:|------------------:|------------------:|-----------------:|-----------------:|
| prompt   |            117.67 |            18.15 |                19 |               337 |                3 |               51 |
| A        |            164.09 |            26.15 |                 1 |               472 |                1 |               81 |
| B        |            166.62 |            26.52 |                 2 |               662 |                1 |              118 |
| C        |            167.23 |            26.55 |                 2 |               530 |                1 |               82 |
| D        |            162.56 |            26    |                 1 |               450 |                1 |               78 |
| E        |            164.23 |            26.19 |                 2 |               587 

In [11]:
def clean_text(text):
    """
    Base text cleaning function. 
    Designed to be modular so future steps (like regex stripping) can be added here.
    """
    #  Handle missing values safely
    if pd.isna(text) or text is None:
        return ""
    
    #  Convert to string and lowercase
    text = str(text).casefold()
    
    #  Remove extra whitespace (newlines, tabs, and multiple consecutive spaces)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def preprocess_mcq_data(df, text_columns):
    """
    Applies the clean_text function to all specified columns in the dataframe.
    Returns a new dataframe to avoid mutating the original data.
    """
    df_clean = df.copy()
    
    for col in text_columns:
        df_clean[col] = df_clean[col].apply(clean_text)
        
    return df_clean



# Apply the preprocessing
df_cleaned = preprocess_mcq_data(df, text_cols)

# Verify the changes on the first row
print("Before cleaning:")
print(df.loc[5, 'prompt'])
print("\nAfter cleaning:")
print(df_cleaned.loc[5, 'prompt'])

Before cleaning:
Identify the correct statement: What is the effect generated by a spinning superconductor? among the listed options.

After cleaning:
identify the correct statement: what is the effect generated by a spinning superconductor? among the listed options.


In [12]:
import pandas as pd
options_cols = ['A', 'B', 'C', 'D', 'E']

print("---  PROMPT VS. OPTIONS LENGTH (Word Count) ---")
# Calculate average word count for prompts
avg_prompt_len = df_cleaned['prompt'].str.split().str.len().mean()

# Calculate average word count across all 5 options combined
avg_option_len = df_cleaned[options_cols].apply(lambda x: x.str.split().str.len()).mean().mean()

print(f"Average Prompt length: {avg_prompt_len:.1f} words")
print(f"Average Option length: {avg_option_len:.1f} words")
print(f"Ratio: Prompts are roughly {avg_prompt_len/avg_option_len:.1f}x longer than options.\n")


print("--- 3. VOCABULARY VARIATION (Effect of Lowercasing) ---")
def get_vocab_size(series):
    # Joins all rows into one giant string, splits by space, and counts unique words
    return len(set(' '.join(series.astype(str)).split()))

# Compare raw vocabulary vs lowercased vocabulary on the prompts
raw_vocab_size = get_vocab_size(df['prompt'])
lower_vocab_size = get_vocab_size(df['prompt'].str.lower())
reduction = raw_vocab_size - lower_vocab_size
percent_reduction = (reduction / raw_vocab_size) * 100

print(f"Original unique words: {raw_vocab_size}")
print(f"Lowercased unique words: {lower_vocab_size}")
print(f"Conclusion: Lowercasing reduced the vocabulary size by {reduction} words ({percent_reduction:.1f}%).\n")


print("--- 4. FORMATTING NORMALIZATION (Effect of Whitespace Removal) ---")
# Count total characters before and after cleaning to see how much "invisible" noise was removed
raw_chars = df[text_cols].astype(str).apply(lambda x: x.str.len()).sum().sum()
clean_chars = df_cleaned[text_cols].astype(str).apply(lambda x: x.str.len()).sum().sum()
chars_removed = raw_chars - clean_chars

print(f"Total characters before cleaning: {raw_chars}")
print(f"Total characters after cleaning: {clean_chars}")
print(f"Conclusion: Removed {chars_removed} unnecessary whitespace characters across the dataset.")

---  PROMPT VS. OPTIONS LENGTH (Word Count) ---
Average Prompt length: 18.1 words
Average Option length: 26.3 words
Ratio: Prompts are roughly 0.7x longer than options.

--- 3. VOCABULARY VARIATION (Effect of Lowercasing) ---
Original unique words: 938
Lowercased unique words: 931
Conclusion: Lowercasing reduced the vocabulary size by 7 words (0.7%).

--- 4. FORMATTING NORMALIZATION (Effect of Whitespace Removal) ---
Total characters before cleaning: 1884796
Total characters after cleaning: 1884796
Conclusion: Removed 0 unnecessary whitespace characters across the dataset.


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

corpus = (
    df_cleaned['prompt'] + " " + 
    df_cleaned['A'] + " " + 
    df_cleaned['B'] + " " + 
    df_cleaned['C'] + " " + 
    df_cleaned['D'] + " " + 
    df_cleaned['E']
)

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(corpus)

print("--- VOCABULARY & SHAPE ---")
vocab_size = len(tfidf.vocabulary_)
print(f"Vocabulary Size (Unique words): {vocab_size}")
print(f"Matrix Shape (Documents x Words): {tfidf_matrix.shape}")

print("\n---  SPARSITY ---")
# Calculate how much of the matrix is filled with zeros
total_elements = tfidf_matrix.shape[0] * tfidf_matrix.shape[1]
non_zero_elements = tfidf_matrix.nnz
zero_elements = total_elements - non_zero_elements
sparsity = (zero_elements / total_elements) * 100

print(f"Non-zero elements: {non_zero_elements}")
print(f"Matrix Sparsity: {sparsity:.2f}% zeros")

print("\n---  FEATURE NAMES ---")
feature_names = tfidf.get_feature_names_out()
print(f"First 10 features: {feature_names[:10]}")

print("\n--- 4. EXAMPLE VECTOR (Row 0) ---")
# Extract the first document's vector
row_0 = tfidf_matrix[0]

# .nonzero() returns a tuple of (row_indices, col_indices). We want the columns.
non_zero_indices = row_0.nonzero()[1]

print(f"Words actually present in Document 0: {len(non_zero_indices)}")
print("Top 5 words by TF-IDF importance in Document 0:")

# Map the non-zero indices back to words and their specific TF-IDF scores
doc_0_features = [(feature_names[idx], row_0[0, idx]) for idx in non_zero_indices]

# Sort them by TF-IDF score in descending order to see what the algorithm found most "important"
doc_0_features_sorted = sorted(doc_0_features, key=lambda x: x[1], reverse=True)

for word, score in doc_0_features_sorted[:5]:
    print(f" - {word}: {score:.4f}")

# --- STEP 4: Common vs. Specific Words (Checking the math) ---

# Choose the same sample document (Row 0)
sample_idx = 0
sample_vector = tfidf_matrix[sample_idx]

# Words we want to investigate
common_words = ['the', 'is', 'what']

print("--- 5. COMMON WORD WEIGHTS (Row 0) ---")
for word in common_words:
    # Check if the word exists in our learned vocabulary
    if word in tfidf.vocabulary_:
        # Get the column index for this word
        word_idx = tfidf.vocabulary_[word]
        # Get the TF-IDF score for this word in Document 0
        score = sample_vector[0, word_idx]
        print(f"Score for '{word}': {score:.4f}")
    else:
        print(f"'{word}' not found in vocabulary.")

print("\n--- COMPARISON ---")
print("Notice how the scores for 'the', 'is', and 'what' compare to the top 5 words")
print("we printed earlier in Step 3. Even if the word 'the' appears 5 times in the ")
print("document and a specific word appears only once, the specific word gets a ")
print("higher score because 'the' is penalized for appearing in every other row.")

--- VOCABULARY & SHAPE ---
Vocabulary Size (Unique words): 2940
Matrix Shape (Documents x Words): (2000, 2940)

---  SPARSITY ---
Non-zero elements: 94219
Matrix Sparsity: 98.40% zeros

---  FEATURE NAMES ---
First 10 features: ['000' '013' '013343' '02m' '10' '100' '1012' '1036' '11' '12']

--- 4. EXAMPLE VECTOR (Row 0) ---
Words actually present in Document 0: 90
Top 5 words by TF-IDF importance in Document 0:
 - martin: 0.3382
 - heidegger: 0.3382
 - human: 0.2818
 - future: 0.2818
 - past: 0.2539
--- 5. COMMON WORD WEIGHTS (Row 0) ---
Score for 'the': 0.1852
Score for 'is': 0.0820
Score for 'what': 0.0102

--- COMPARISON ---
Notice how the scores for 'the', 'is', and 'what' compare to the top 5 words
we printed earlier in Step 3. Even if the word 'the' appears 5 times in the 
document and a specific word appears only once, the specific word gets a 
higher score because 'the' is penalized for appearing in every other row.


In [14]:
from sklearn.metrics.pairwise import cosine_similarity

# --- Transform all columns into TF-IDF vectors ---
vec_prompt = tfidf.transform(df_cleaned['prompt'])
vec_A = tfidf.transform(df_cleaned['A'])
vec_B = tfidf.transform(df_cleaned['B'])
vec_C = tfidf.transform(df_cleaned['C'])
vec_D = tfidf.transform(df_cleaned['D'])
vec_E = tfidf.transform(df_cleaned['E'])

# --- Compute Cosine Similarity using sklearn ---
# cosine_similarity returns an NxN matrix. 
# We take .diagonal() to compare row i of prompts only with row i of the options.
sim_A = cosine_similarity(vec_prompt, vec_A).diagonal()
sim_B = cosine_similarity(vec_prompt, vec_B).diagonal()
sim_C = cosine_similarity(vec_prompt, vec_C).diagonal()
sim_D = cosine_similarity(vec_prompt, vec_D).diagonal()
sim_E = cosine_similarity(vec_prompt, vec_E).diagonal()

predictions = []
top_1_predictions = []
all_scores = []
correct_sims = []
wrong_sims_avg = []

for i in range(len(df_cleaned)):
    scores = {
        'A': sim_A[i], 
        'B': sim_B[i], 
        'C': sim_C[i], 
        'D': sim_D[i], 
        'E': sim_E[i]
    }
    
    sorted_opts = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    top_3 = [opt[0] for opt in sorted_opts[:3]]
    
    predictions.append(" ".join(top_3))
    top_1_predictions.append(top_3[0])
    all_scores.append(scores)
    
    # Track similarity for the Correct Answer vs Wrong Answers
    correct_ans = df.iloc[i]['answer']
    correct_sims.append(scores[correct_ans])
    
    # Calculate the mean similarity of the 4 incorrect options
    wrong_scores = [score for opt, score in scores.items() if opt != correct_ans]
    wrong_sims_avg.append(np.mean(wrong_scores))


df_cleaned['Prediction_Top3'] = predictions
df_cleaned['Prediction_Top1'] = top_1_predictions
df_cleaned['Correct_Sim'] = correct_sims
df_cleaned['Wrong_Sim_Avg'] = wrong_sims_avg

In [15]:
# --- EVALUATION METRICS ---
def calculate_map_at_3(predictions_series, actual_series):
    score = 0.0
    for pred_str, actual in zip(predictions_series, actual_series):
        pred_list = pred_str.split()
        if actual in pred_list:
            rank = pred_list.index(actual) + 1
            score += 1.0 / rank # 1 for 1st, 0.5 for 2nd, 0.33 for 3rd
    return score / len(predictions_series)

map3_score = calculate_map_at_3(df_cleaned['Prediction_Top3'], df['answer'])
top1_accuracy = (df_cleaned['Prediction_Top1'] == df['answer']).mean()

print(f"MAP@3 Score: {map3_score:.4f}")
print(f"Top-1 Accuracy: {top1_accuracy:.2%}")

# --- CORRECT VS WRONG SIMILARITY ---
print(f"\nAvg Similarity (Correct Option): {np.mean(correct_sims):.4f}")
print(f"Avg Similarity (Wrong Options):  {np.mean(wrong_sims_avg):.4f}")

# --- DETAILED INSPECTION ---
def print_breakdown(idx, title):
    scores = all_scores[idx]
    sorted_opts = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    correct_ans = df.iloc[idx]['answer']
    
    print(f"\n--- {title} (Row {idx}) ---")
    print(f"Prompt: {df.iloc[idx]['prompt']}")
    print(f"Correct Answer: Option {correct_ans}")
    print(f"Cosine(correct): {scores[correct_ans]:.4f}")
    print(f"Highest cosine:  Option {sorted_opts[0][0]} ({sorted_opts[0][1]:.4f})")
    print(f"Second highest:  Option {sorted_opts[1][0]} ({sorted_opts[1][1]:.4f})")
    print(f"Third highest:   Option {sorted_opts[2][0]} ({sorted_opts[2][1]:.4f})")

# Get indices for successes and failures
success_indices = df_cleaned[df_cleaned['Prediction_Top1'] == df['answer']].index.tolist()
failure_indices = df_cleaned[~df_cleaned.apply(lambda row: row['answer'] in row['Prediction_Top3'].split(), axis=1)].index.tolist()

print("\n" + "="*50)
print("🔍 INSPECTING SUCCESSES")
print("Look for: Definition questions, exact keyword matching, and factual recall.")
for i in success_indices[:3]:  # Print first 3 successes
    print_breakdown(i, "SUCCESS")

print("\n" + "="*50)
print("🔍 INSPECTING FAILURES")
print("Look for: Multi-hop reasoning, synonyms (no keyword overlap), or negation ('Which is NOT...').")
for i in failure_indices[:3]:  # Print first 3 failures
    print_breakdown(i, "FAILURE")

MAP@3 Score: 0.2859
Top-1 Accuracy: 14.40%

Avg Similarity (Correct Option): 0.2967
Avg Similarity (Wrong Options):  0.3138

🔍 INSPECTING SUCCESSES
Look for: Definition questions, exact keyword matching, and factual recall.

--- SUCCESS (Row 5) ---
Prompt: Identify the correct statement: What is the effect generated by a spinning superconductor? among the listed options.
Correct Answer: Option C
Cosine(correct): 0.0214
Highest cosine:  Option C (0.0214)
Second highest:  Option E (0.0212)
Third highest:   Option B (0.0211)

--- SUCCESS (Row 9) ---
Prompt: Which of the following is correct? What is the relationship between the Wigner function and the density matrix operator? from the following choices.
Correct Answer: Option A
Cosine(correct): 0.7222
Highest cosine:  Option A (0.7222)
Second highest:  Option C (0.6413)
Third highest:   Option E (0.6308)

--- SUCCESS (Row 12) ---
Prompt: Choose the correct answer: Which of the following statements accurately describes the relationship bet

In [16]:
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity

# We use a fast, pretrained model for the baseline. 
# You can change this to 'word2vec-google-news-300' for higher accuracy later.
print("Loading pretrained Word2Vec model")
w2v_model = api.load('glove-wiki-gigaword-50') 

def get_sentence_embedding(text, model):
    """
    Converts a sentence into a single dense vector by averaging its word embeddings.
    """
    words = str(text).split()
    # Keep only words that exist in the model's vocabulary
    valid_words = [word for word in words if word in model]
    
    if not valid_words:
        # If no words are found, return a zero vector of the correct dimension
        return np.zeros(model.vector_size)
    
    # Extract vectors and compute the mean along the column axis
    vectors = np.array([model[word] for word in valid_words])
    return vectors.mean(axis=0)

print("Converting text to average word embeddings...")

vec_prompt_w2v = np.vstack(df_cleaned['prompt'].apply(lambda x: get_sentence_embedding(x, w2v_model)))
vec_A_w2v = np.vstack(df_cleaned['A'].apply(lambda x: get_sentence_embedding(x, w2v_model)))
vec_B_w2v = np.vstack(df_cleaned['B'].apply(lambda x: get_sentence_embedding(x, w2v_model)))
vec_C_w2v = np.vstack(df_cleaned['C'].apply(lambda x: get_sentence_embedding(x, w2v_model)))
vec_D_w2v = np.vstack(df_cleaned['D'].apply(lambda x: get_sentence_embedding(x, w2v_model)))
vec_E_w2v = np.vstack(df_cleaned['E'].apply(lambda x: get_sentence_embedding(x, w2v_model)))


sim_A_w2v = cosine_similarity(vec_prompt_w2v, vec_A_w2v).diagonal()
sim_B_w2v = cosine_similarity(vec_prompt_w2v, vec_B_w2v).diagonal()
sim_C_w2v = cosine_similarity(vec_prompt_w2v, vec_C_w2v).diagonal()
sim_D_w2v = cosine_similarity(vec_prompt_w2v, vec_D_w2v).diagonal()
sim_E_w2v = cosine_similarity(vec_prompt_w2v, vec_E_w2v).diagonal()


predictions_w2v = []
top_1_predictions_w2v = []

for i in range(len(df_cleaned)):
    scores = {
        'A': sim_A_w2v[i], 'B': sim_B_w2v[i], 
        'C': sim_C_w2v[i], 'D': sim_D_w2v[i], 'E': sim_E_w2v[i]
    }
    sorted_opts = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    top_3 = [opt[0] for opt in sorted_opts[:3]]
    
    predictions_w2v.append(" ".join(top_3))
    top_1_predictions_w2v.append(top_3[0])

# Add to dataframe for comparison
df_cleaned['Prediction_W2V_Top3'] = predictions_w2v
df_cleaned['Prediction_W2V_Top1'] = top_1_predictions_w2v

# Calculate Metrics
map3_w2v = calculate_map_at_3(df_cleaned['Prediction_W2V_Top3'], df['answer'])
top1_w2v = (df_cleaned['Prediction_W2V_Top1'] == df['answer']).mean()

print("\n--- PHASE 5 EVALUATION ---")
print(f"Word2Vec MAP@3 Score: {map3_w2v:.4f} (vs TF-IDF: {map3_score:.4f})")
print(f"Word2Vec Top-1 Accuracy: {top1_w2v:.2%} (vs TF-IDF: {top1_accuracy:.2%})")

Loading pretrained Word2Vec model
[==================================================] 100.0% 66.0/66.0MB downloaded
Converting text to average word embeddings...

--- PHASE 5 EVALUATION ---
Word2Vec MAP@3 Score: 0.3089 (vs TF-IDF: 0.2859)
Word2Vec Top-1 Accuracy: 12.95% (vs TF-IDF: 14.40%)


In [17]:
# ---  OVERALL COMPARISON ---
# Count how many each got completely right (Top-1)
tfidf_correct_mask = df_cleaned['Prediction_Top1'] == df['answer']
w2v_correct_mask = df_cleaned['Prediction_W2V_Top1'] == df['answer']

print("---  OVERALL WINS ---")
print(f"TF-IDF Correct (Top-1): {tfidf_correct_mask.sum()}")
print(f"Word2Vec Correct (Top-1): {w2v_correct_mask.sum()}")

---  OVERALL WINS ---
TF-IDF Correct (Top-1): 288
Word2Vec Correct (Top-1): 259


In [18]:
# ---  WHICH QUESTIONS IMPROVED? (Word2Vec wins, TF-IDF loses) ---
# Find rows where Word2Vec got Top-1 correct, but TF-IDF didn't even get it in Top-3
w2v_improved_mask = (df_cleaned['Prediction_W2V_Top1'] == df['answer']) & \
                    (~df_cleaned.apply(lambda row: row['answer'] in str(row['Prediction_Top3']).split(), axis=1))

df_improved = df_cleaned[w2v_improved_mask]
print(f"\n---  W2V IMPROVEMENTS ({len(df_improved)} questions) ---")
if not df_improved.empty:
    for i in range(min(2, len(df_improved))): # Print up to 2 examples
        row = df_improved.iloc[i]
        print(f"\nPrompt: {row['prompt']}")
        print(f"Correct Answer ({row['answer']}): {row[row['answer']]}")
        print(f"TF-IDF Guessed: {row['Prediction_Top1']} | Word2Vec Guessed: {row['Prediction_W2V_Top1']}")


---  W2V IMPROVEMENTS (108 questions) ---

Prompt: choose the correct answer: what is the relationship between mass, force, and acceleration, according to sir isaac newton's laws of motion?
Correct Answer (D): mass is an inertial property that determines an object's tendency to remain at constant velocity unless acted upon by an outside force. according to newton's laws of motion and the formula f = ma, an object with a mass of one kilogram accelerates at one meter per second per second when acted upon by a force of one newton.
TF-IDF Guessed: A | Word2Vec Guessed: D

Prompt: choose the correct answer: what is the reason for recycling rare metals according to the united nations?
Correct Answer (E): the demand for rare metals will quickly exceed the consumed tonnage in 2013, and recycling rare metals with a worldwide production lower than 100 000 t/year is urgent and priority should be placed on it in order to conserve natural resources and energy.
TF-IDF Guessed: B | Word2Vec Guessed:

In [19]:
# ---  WHICH QUESTIONS REGRESSED? (TF-IDF wins, Word2Vec loses) ---
# Find rows where TF-IDF got Top-1 correct, but Word2Vec failed
tfidf_wins_mask = (df_cleaned['Prediction_Top1'] == df['answer']) & \
                  (~df_cleaned.apply(lambda row: row['answer'] in str(row['Prediction_W2V_Top3']).split(), axis=1))

df_regressed = df_cleaned[tfidf_wins_mask]
print(f"\n---  TF-IDF WINS ({len(df_regressed)} questions) ---")
if not df_regressed.empty:
    for i in range(min(2, len(df_regressed))):
        row = df_regressed.iloc[i]
        print(f"\nPrompt: {row['prompt']}")
        print(f"Correct Answer ({row['answer']}): {row[row['answer']]}")
        print(f"TF-IDF Guessed: {row['Prediction_Top1']} | Word2Vec Guessed: {row['Prediction_W2V_Top1']}")

# ---  WHICH QUESTIONS REMAIN DIFFICULT? (Both fail) ---
both_fail_mask = (~df_cleaned.apply(lambda row: row['answer'] in str(row['Prediction_Top3']).split(), axis=1)) & \
                 (~df_cleaned.apply(lambda row: row['answer'] in str(row['Prediction_W2V_Top3']).split(), axis=1))

df_hard = df_cleaned[both_fail_mask]
print(f"\n---  HARD QUESTIONS ({len(df_hard)} questions) ---")
print("Look for: Multi-step reasoning, logic puzzles, or 'NOT/Except' questions.")
if not df_hard.empty:
    for i in range(min(2, len(df_hard))):
        row = df_hard.iloc[i]
        print(f"\nPrompt: {row['prompt']}")
        print(f"Correct Answer ({row['answer']}): {row[row['answer']]}")
        print(f"TF-IDF Guessed: {row['Prediction_Top1']} | Word2Vec Guessed: {row['Prediction_W2V_Top1']}")
    


---  TF-IDF WINS (79 questions) ---

Prompt: which of the following is correct? what is the relationship between the wigner function and the density matrix operator? from the following choices.
Correct Answer (A): the wigner function w(x, p) is the wigner transform of the density matrix operator ρ̂, and the trace of an operator with the density matrix wigner-transforms to the equivalent phase-space integral overlap of g(x,p) with the wigner function.
TF-IDF Guessed: A | Word2Vec Guessed: E

Prompt: choose the correct answer: which of the following statements accurately describes the relationship between the dimensions of a diffracting object and the angular spacing of features in the diffraction pattern? carefully.
Correct Answer (D): the angular spacing of features in the diffraction pattern is inversely proportional to the dimensions of the object causing the diffraction. therefore, if the diffracting object is smaller, the resulting diffraction pattern will be wider.
TF-IDF Guessed

In [20]:
test_cleaned = preprocess_mcq_data(test_df, text_cols)

# Convert test text to sentence embeddings
test_prompt_w2v = np.vstack(
    test_cleaned["prompt"].apply(lambda x: get_sentence_embedding(x, w2v_model))
)

test_A_w2v = np.vstack(
    test_cleaned["A"].apply(lambda x: get_sentence_embedding(x, w2v_model))
)

test_B_w2v = np.vstack(
    test_cleaned["B"].apply(lambda x: get_sentence_embedding(x, w2v_model))
)

test_C_w2v = np.vstack(
    test_cleaned["C"].apply(lambda x: get_sentence_embedding(x, w2v_model))
)

test_D_w2v = np.vstack(
    test_cleaned["D"].apply(lambda x: get_sentence_embedding(x, w2v_model))
)

test_E_w2v = np.vstack(
    test_cleaned["E"].apply(lambda x: get_sentence_embedding(x, w2v_model))
)


sim_A = cosine_similarity(test_prompt_w2v, test_A_w2v).diagonal()
sim_B = cosine_similarity(test_prompt_w2v, test_B_w2v).diagonal()
sim_C = cosine_similarity(test_prompt_w2v, test_C_w2v).diagonal()
sim_D = cosine_similarity(test_prompt_w2v, test_D_w2v).diagonal()
sim_E = cosine_similarity(test_prompt_w2v, test_E_w2v).diagonal()

predictions = []

for i in range(len(test_cleaned)):
    scores = {
        "A": sim_A[i],
        "B": sim_B[i],
        "C": sim_C[i],
        "D": sim_D[i],
        "E": sim_E[i],
    }

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    top3 = [label for label, _ in ranked[:3]]

    predictions.append(" ".join(top3))

In [21]:
submission = test_df[["id"]].copy()

submission["Prediction"] = predictions

submission.head()

submission.to_csv("submission_word2vec_baseline.csv", index=False)

print("Submission file saved successfully!")

Submission file saved successfully!
